## Attention

Sources

- https://arxiv.org/pdf/1911.02150
- https://github.com/QasimWani/simple-transformer

In this file we'll go through

- basic einsum knowledge
- basic einops knowledge (rearrange)
- dot product attention
- MHA
- GQA / MQA
- MLA
- KV cache implementation with all of these
- Linear attention


Pytorch syntax


Familiarize yourself with einsum notation - this is the standard of how matrices are multiplied in production


In [23]:
import torch
import math
import torch.nn.functional as F 

A = torch.tensor([[1,2],[3,4]])
B = torch.tensor([[1,1],[1,1]])
C = torch.tensor([1,2,3,4])
D = torch.tensor([1,1,1,1])

# dot product - uv 
dot = torch.einsum('i,i->', C, D) # 2 inputs, both 1D denoted by i, input dimensions are i and outputs has no dimensions (scalar)

# outer product - uv^T
outer = torch.einsum('i,j -> ij', C, D) # CD^T

# matmul
matmul = torch.einsum('ac, cd -> ad', A, B)

# matmul with no summing
sumless_matmul = torch.einsum('ac,cd -> acd', A, B) # inner dimension that is hidden dim persists (2,2,2) where sumless[0][1][1] = A[0][1] * B[1][1] for example is the separated doc product of C[0][1]
sumless_matmul # multiplies the corresponding values in A and B, but does not sum them
summed_matmul = torch.einsum('acd->ad',sumless_matmul)
summed_matmul

# batched matmul
input = torch.rand(32, 1024, 64) # normal batch size embedding dim of 64
hidden = torch.rand(64, 256)
output = torch.einsum('ijk, kl -> ijl', input, hidden)
output.shape # torch.Size([32, 1024, 256]), nice!

# full quadratic attention
query = torch.rand(32, 1024, 256)
key = torch.rand(32, 1024, 256)
value = torch.rand(32, 1024, 256)

qk = torch.einsum('bqd, bkd -> bqk', query, key)
d = query.shape[-1]
softmax_qk = F.softmax(qk/math.sqrt(d), dim=-1) # softmax, consuming the k dimension (so row-wise softmax)
attention = torch.einsum('bqk, bkd -> bkd', softmax_qk, value)

# without einsums
qk = query @ key.transpose(-2, -1) / math.sqrt(query.shape[-1])
B, T, T = qk.shape
mask = torch.tril(torch.ones(T, T)).bool()
print(mask.shape, qk.shape)
masked_qk = qk.masked_fill(~mask, float('-inf')) # where the mask is False, put -inf
print(masked_qk)

qkv = F.softmax(qk, dim=-1) @ value
qkv = qk @ value # (B, T, T) @ (T, d) = (B, T, d)

assert attention.shape == qkv.shape

torch.Size([1024, 1024]) torch.Size([32, 1024, 1024])
tensor([[[3.9365,   -inf,   -inf,  ...,   -inf,   -inf,   -inf],
         [4.0743, 3.8907,   -inf,  ...,   -inf,   -inf,   -inf],
         [4.0541, 3.7908, 3.9739,  ...,   -inf,   -inf,   -inf],
         ...,
         [4.0011, 3.9469, 3.9740,  ..., 4.0106,   -inf,   -inf],
         [4.0143, 3.6939, 4.1044,  ..., 4.0001, 4.0864,   -inf],
         [4.1160, 3.8555, 4.3491,  ..., 4.2996, 4.2417, 4.0130]],

        [[4.2850,   -inf,   -inf,  ...,   -inf,   -inf,   -inf],
         [4.1208, 4.1571,   -inf,  ...,   -inf,   -inf,   -inf],
         [4.1241, 4.0620, 3.6656,  ...,   -inf,   -inf,   -inf],
         ...,
         [4.0660, 4.0712, 3.9484,  ..., 4.1646,   -inf,   -inf],
         [3.9658, 4.0996, 3.7159,  ..., 3.9716, 3.7293,   -inf],
         [4.1847, 4.1687, 4.0127,  ..., 4.1992, 4.0653, 4.4096]],

        [[4.2618,   -inf,   -inf,  ...,   -inf,   -inf,   -inf],
         [4.2777, 3.7168,   -inf,  ...,   -inf,   -inf,   -inf],
    

Dot product attention


In [ ]:
# normal dot product attention
import torch
import numpy as np
import torch.nn.functional as F

def DotProductAttention(Q, K, V):

	"""
	Args
	- Q = matrix of queries
	- K = matrix of keys
	- V = matrix of values

	Reasoning about time complexity: 
	- (b, t, t) * (b, t, d) -> (b, t, d), summing t rows of t terms with a column of t terms (t sums), over d columns
	- time complexity = t^2 * d

	Time complexity: O(batch * seq_len^2 * dim + batch * seq_len * dim)
	Space complexity: O(batch * seq_len * d + batch * seq_len^2) # 3 qkv matrices, then the attention matrix - removing constants

	Auxiliary space complexity - the space allocated to the algorithm during computation, not including inputs
	Input space complexity - the space allocated to the inputs + the intermediary vectors, tensors, etc. needed by the algorithm

	e.g. the auxiliary space complexity of single head attention is batch * seq_len^2
	"""

	b, t, d = Q.shape

	# scale the raw scores first, then mask, then softmax
	qk = torch.einsum('bqd, bkd -> bqk', Q, K) / (d**0.5)

	# causal masking - must happen BEFORE the softmax so each row renormalises
	# over only the positions it is allowed to see
	mask = torch.tril(torch.ones(t, t, device=Q.device)).bool()
	masked_qk = qk.masked_fill(~mask, float('-inf'))

	softmax_qk = F.softmax(masked_qk, dim=-1) # remember you need to softmax along the dim

	qkv = torch.einsum('bqk, bkd -> bqd', softmax_qk, V)

	return qkv

Q = torch.rand(32, 1024, 64)
K = torch.rand(32, 1024, 64)
V = torch.rand(32, 1024, 64)

# the reference must also be causal, otherwise we are comparing different maths
torch.allclose(DotProductAttention(Q, K, V),
               F.scaled_dot_product_attention(Q, K, V, is_causal=True),
               atol=1e-6)

_Quick primer on einops_

**Simple flatten**  
rearrange(query, 'b q (n d) -> b q h') # this flattens along the last 2 dimensions

- (32, 1024, 8, 32) -> (32, 1024, 256)

**Simple split**  
rearrange(query, 'b q d -> b q (n d)', n=8) # this does the reverse, and splits along the last dimension

- (32, 1024, 256) -> (32, 1024, 8, 32)

**Complex split and transpose**  
rearrange(query, 'b q (n d) -> b n q d', n=8) # first this splits the last dimension into 2 using the parentheses of the input, indicating we should split, then we transpose dim2 and dim1

- (32, 1024, 256) -> (32, 1024, 8, 32) -> (32, 8, 1024, 32)


**Visual intuition on differences between attention implementations**

<img src="https://miro.medium.com/v2/resize:fit:1200/1*oZq5NCt0BZbcSwhCztL-cQ.png" width="500">


MHA


Quick primer on torch operations impact on memory

- torch.reshape = safe for non-contigous tensors, creates a new memory copy or uses view() and modifies the original tensor in memory
- torch.view = only on contiguous, modifies original tensor instead of copying
- torch.cat = concatenates and returns a new tensor, takes more memory
- torch.split = no new tensor, they create a view on the original tensor (metadata, memory pointers)
- torch.chunk - no new tensor, a view on the existing tensor (metadata, memory pointers)

rearrange from einops does view by default, to save memory


In [93]:
w_in = torch.rand(32, 1024, 64)

# MHA with class
import torch.nn as nn
from einops import rearrange

class MHA(nn.Module):

	def __init__(self, d_embed: int=64, n_heads: int=8): 
		super().__init__()
		assert d_embed % n_heads == 0
		self.n_heads = n_heads

		self.qkv = nn.Linear(d_embed, d_embed * 3) # qkv weight matrices
		self.w_out = nn.Linear(d_embed, d_embed) # takes the final att output and projects it back to d_embed

	def forward(self, w_in): 

		B, T, C = w_in.shape

		qkv = self.qkv(w_in) # matmul of the input with the qkv matrix
		
		q, k, v = rearrange(qkv, 'b t (s h d) -> s b h t d', s=3, h=self.n_heads) # split dimensions into 3 for q,k,v and then into n_heads for each head

		d_head = q.shape[-1]
		scores = torch.einsum('b h q d, b h k d -> b h q k', q, k) / (d_head ** 0.5)
		mask = torch.tril(torch.ones(T,T, device=w_in.device)).bool()
		masked_probs = F.softmax(scores.masked_fill(~mask, float('-inf')), dim=-1) # (b, h, q, k)
		values = torch.einsum('b h q k, b h k d -> b h q d', masked_probs, v)
		out = rearrange(values, 'b h v d -> b v (h d)')

		return self.w_out(out)

mha = MHA(d_embed=64, n_heads=8)
out = mha(w_in)
out.shape

torch.Size([32, 1024, 64])

Looped implementation of MHA

- non performant but good for conceptual understanding of MHA


In [ ]:
# multi-head attention

import torch
import numpy as np
import torch.nn.functional as F

def MultiHeadAttention(Q, K, V, n):	

	"""
	Args
	- Q = matrix of queries
	- K = matrix of keys
	- V = matrix of values
	- n = number of heads for query, key and values

	In multi-head attention we want to split the query matrix into multiple heads, and same for the rest, and just multiply heads together

	Each head sees the full sequence but only a D//n slice of the embedding, so the
	per-head scores are still (B, T, T) - only the last dim shrinks.
	"""
	B, T, D = Q.shape
	assert D % n == 0

	chunked_Q = torch.split(Q, D//n, dim=-1) # split along the last dimension
	chunked_K = torch.split(K, D//n, dim=-1)
	chunked_V = torch.split(V, D//n, dim=-1)

	# the causal mask is the same for every head, so build it once
	mask = torch.tril(torch.ones(T, T, device=Q.device)).bool()

	heads = []
	for q, k, v in zip(chunked_Q, chunked_K, chunked_V):

		b, t, d = q.shape # d is the per-head dim (D//n), which is what we scale by

		qk = torch.einsum('bqd, bkd -> bqk', q, k) / (d**0.5)
		masked_qk = qk.masked_fill(~mask, float('-inf'))
		softmax_qk = F.softmax(masked_qk, dim=-1)

		heads.append(torch.einsum('bqk, bkd -> bqd', softmax_qk, v)) # (B, T, D//n)

	# concatenating n heads of (B, T, D//n) along the last dim rebuilds (B, T, D)
	out = torch.cat(heads, dim=-1)
	return out

Q = torch.rand(32, 1024, 64)
K = torch.rand(32, 1024, 64)
V = torch.rand(32, 1024, 64)

n = 4
B, T, D = Q.shape

# reshape into an explicit head dim so the reference does causal *multi-head*
# attention too - comparing against the single-head default would always fail
to_heads = lambda x: x.view(B, T, n, D//n).transpose(1, 2)
ref = F.scaled_dot_product_attention(to_heads(Q), to_heads(K), to_heads(V), is_causal=True)
ref = ref.transpose(1, 2).reshape(B, T, D)

torch.allclose(MultiHeadAttention(Q, K, V, n), ref, atol=1e-6)

MHA with KV cache implementation


In [ ]:
from einops import rearrange
import torch.nn as nn

class MHA(nn.Module): 
	
	def __init__(self, d_embed: int, num_heads: int):
		super().__init__()
		assert d_embed % num_heads == 0
		self.num_heads = num_heads
		self.qkv = nn.Linear(d_embed, d_embed*3)
		self.proj = nn.Linear(d_embed, d_embed)

	def forward(self, x_in, block_kv_cache=None, attention_mask=None):

		is_prefill = block_kv_cache is None
		
		B, T, C = x_in.shape # single token query, (B, 1, C)

		qkv = self.qkv(x_in)
		query, key, value = rearrange(qkv, 'b t (s n d) -> s b n t d', n=self.num_heads, s=3) # split into the query key and value matrices
		d_head = query.shape[-1]

		if is_prefill is False:
			key = torch.cat((block_kv_cache['key'], key), dim=2) # concat to the t x d
			value = torch.cat((block_kv_cache['value'], value), dim=2)
		
		qk = torch.einsum('b n q d, b n k d -> b n q k', query, key) / (d_head**0.5) # scale down before masking so you're not dividing float('-inf') by sqrt(d_head)
		
		if is_prefill:
			# causal masking 
			mask = torch.tril(torch.ones(T,T, device=x_in.device)).bool() # causal masking, remember to move the causal mask to device
			qk = qk.masked_fill(~mask, float('-inf'))

		scores = F.softmax(qk, dim=-1) # softmax across the columns, per row

		out = torch.einsum('b n q k, b n k d -> b n q d', scores, value)
		concat_out = rearrange(out, 'b n q d -> b q (n d)')
		attention_output = self.proj(concat_out)

		block_kv_cache = {
			'key': key,
			'value': value
		}

		return attention_output, block_kv_cache

mha = MHA(d_embed=256, num_heads=8)
x = torch.rand(32, 10, 256)

full, _ = mha(x)                            # prefill all 10 at once

out, cache = mha(x[:, :1])                  # prefill 1 token
outs = [out]
for t in range(1, 10):
    out, cache = mha(x[:, t:t+1], cache)    # decode one at a time
    outs.append(out)

print(torch.allclose(full, torch.cat(outs, dim=1), atol=1e-5))   # should be True

# learnings: 
# - remember to move to device when creating a net new tensor (assume x_in is already on device)
# - basically for any of these: torch.ones, torch.zeros, torch.arange, torch.rand, torch.eye, torch.tensor, torch.full, torch.linspace
# - scale -> mask -> softmax

True


GQA with KV cache

- recall MQA is just GQA where num_groups = 1
- better masking - instead of materializing a new mask for every transformer block, we just materialize a single buffer (here we are coding an individual attention block so this is not so relevant, but if we were coding class GPT that is the high level class, this would save a lot of memory by allocating for just one mask-sized matrix in VRAM)
- buffers are also registered on CUDA, instead of CPU


In [ ]:
class GQA(nn.Module):
	"""
	Keep num_heads for query heads, but for KV heads we'll only have num_groups of each, with each group being d_embed // num_groups, so the heads are wider, but there are less of them
	- space complexity comparison for KV: 
		- MHA: 2 * num_heads * (d_embed // num_heads) = 2 * 8 * (256/8)
		- GQA: 2 * num_groups * (d_embed // num_heads) = 2 * 2 * (256/8)
		- note GQA == MHA if num_groups = num_heads! 
	- time complexity comparison for KV
		- MHA: O(N^2 * d) 
		- GQA: same
	"""

	def __init__(self, d_embed:int, num_heads:int, num_groups:int, max_seq_len:int):
		super().__init__()
		assert num_heads % num_groups == 0 # otherwise we cannot replicate the group heads evenly to attend to the query heads
		
		self.num_heads: int = num_heads
		self.num_groups = num_groups
		self.head_dim = d_embed // num_heads # e.g. 64 embed_dim, 8 query heads, 8 wide head_dim - we then replicate them to attend to each query head
		self.query = nn.Linear(d_embed, d_embed)
		self.key = nn.Linear(d_embed, num_groups * self.head_dim)
		self.value = nn.Linear(d_embed, num_groups * self.head_dim)

		self.w_out = nn.Linear(d_embed, d_embed) # mixes the heads together
		
		self.register_buffer('causal_mask', torch.tril(torch.ones(max_seq_len, max_seq_len)).bool(), persistent=False) # persistence means not saved in state_dict() when checkpointing or in model.parameters. This is correct - prevents gradient accum and also resets for every new run when loading the checkpoint again

	def forward(self, x_in, attention_mask=False, block_kv_cache=None): 
		is_prefill = block_kv_cache is None

		b, t, d = x_in.shape

		query = self.query(x_in) # b, t, d
		key = self.key(x_in) # batch, seq_len, dims
		value = self.value(x_in) # batch, seq_len, dims

		query = rearrange(query, 'b t (n d) -> b n t d', n=self.num_heads) # b, num_heads, seq_len, head_dim
		
		if not is_prefill: 
			key = torch.cat((block_kv_cache['key'], key), dim=1) # b k d, we want to concatenate along the (k,d) dimension
			value = torch.cat((block_kv_cache['value'], value), dim=1)

		### NOTE: the flaw with this implementation below is that it materializes the full KV attention heads in memory via the flattening operation 
		# i.e. the full heads matrcies: b n q d and b n q k are saved which would be the same as MHA

		# num_repeats = num_heads // num_groups
		# key = rearrange(key, 'b t (n d) -> b n 1 t d', n=self.num_groups).expand(b, self.num_repeats, self.num_groups, t, head_dim).flatten(1,2) # b, num_heads, seq_len, head_dim
		# value = rearrange(value, 'b t (n d) -> b n 1 t d', n=self.num_groups).expand(b, self.num_repeats, self.num_groups, t, head_dim).flatten(1,2) # b, num_heads, seq_len, head_dim
		# logits = torch.einsum('b n q d, b n k q -> b n q k', query, key) / (d ** 0.5)

		# NOTE: for Qasim: using b g r t d vs. b r g t d changes the q @ k multiplication, where r = num_repeats and g = num_groups
		# b g r t d -> replicate each group r times, say r =3 
		# dim1, dim2 looks like [g1, g1, g1], [g2, g2, g2], ...

		# while b r g t d...
		# dim1, dim2 looks like [g1, g2, g3, ...], [g1, g2, g3, ...], [g1, g2, g3, ...]

		# when we then do broadcasting you get: 
		# query heads index 1 -> num_heads//num_groups attending to g1, g2, g3,... instead of just g1!
		# this is not the original implementation, but if you start from pretraining like this - then it ends up being ok. But when loading weights, it would break the model

		### use broadcasting instead, which doesn't materialize any matrices
		q = rearrange(query, 'b (g r) q d -> b g r q d', g=self.num_groups) # go from batch * num_heads * seq_len * dim -> batch * num_groups * num_repeats * seq_len * dim_groups
		k = rearrange(key, 'b k (g d) -> b g k d', g=self.num_groups)
		v = rearrange(value, 'b v (g d) -> b g v d', g=self.num_groups) 
		logits = torch.einsum('b g r q d, b g k d -> b g r q k', q, k) / (self.head_dim**0.5)# multiplying along the dim=3, dim=4 dimensions, broadcasting along dim=2
		
		if is_prefill and attention_mask:
			logits = logits.masked_fill(~self.causal_mask[:t, :t], float('-inf'))

		scores = F.softmax(logits, dim=-1)

		attention = torch.einsum('b g r q k, b g k d -> b g r q d', scores, v) # broadcasts the value matrix along dim2 (32, 4, 2, 1024, 256) @ (32, 4, 1024, 256)
		attention = rearrange(attention, 'b g r q d -> b q (g r d)')
		
		attention_output = self.w_out(attention) # b v d

		block_kv_cache = {
			"key": key,
			"value": value,
		}

		return attention_output, block_kv_cache

x = torch.rand(32, 10, 256)
b, seq_len, d = x.shape

full, _ = gqa(x, attention_mask=True)
# out, cache = gqa(x[:, :1])
outs = []
cache=None
for t in range(seq_len): # range 10, going from token 0->9
    out, cache = gqa(x[:, t:t+1], block_kv_cache=cache)
    outs.append(out)

print(torch.allclose(full, torch.cat(outs, dim=1), atol=1e-5))   # should be True


True


## MLA with KV cache

- basic guide to MLA
  - we use low-rank matrices to approximate the full Q, K, V matrices (SVD)
  - so for Q, we have 2 low-rank matrices (down_proj, up_proj), with rank = 12 typically
  - for K, V, they both share the low-rank matrix (down_proj), but have independent up_proj matrices, with rank = 4 typically
- saves memory by using low-rank matrices, similar motivation as LoRA


In [154]:
class MLA(nn.Module): 

	"""
	space complexity
	- MLA: O(2*d*q_rank + d*kv_rank + d*k_rank + d*v_rank) <- just the 5 latent matrices
		- if q_rank = 12, and the rest = 4, d = 64, O(N) = 2,304
	- MHA: O(3*d^2)
		- if d^2 = 64, O(N) = 12,288
	
	As we scale the d_embed and number of attention layers
	KV cache is the full K/V still, so no KV saving

	time complexity (counting composition of QKV matrices): 
	- MLA: O(2 * d * q_rank * seq_len + 2 * k_rank * d * seq_len + 2 * v_rank * d * seq_len + seq_len^2*d + seq_len*d)
	- MHA: O(3*2*d^2*seq_len + seq_len^2*d + seq_len*d)

	MLA gives us both lower time and space complexity
	"""

	def __init__(self, q_latent_dim:int=12, kv_latent_dim:int=4, d_embed:int=256, num_heads:int=8, max_seq_len:int=1024): 
		super().__init__()

		self.q_latent = nn.Linear(d_embed, q_latent_dim)
		self.query = nn.Linear(q_latent_dim, d_embed)
		self.kv_latent = nn.Linear(d_embed, kv_latent_dim)
		self.key = nn.Linear(kv_latent_dim, d_embed)
		self.value = nn.Linear(kv_latent_dim, d_embed)
		self.num_heads = num_heads

		self.w_out = nn.Linear(d_embed, d_embed)
		
		self.register_buffer('causal_mask', torch.tril(torch.ones(max_seq_len, max_seq_len)).bool(), persistent=False) # this is a nn.Module method that registers a self.causal_mask but on CUDA

	def forward(self, x_in:int, attention_mask=False, block_kv_cache=None): 
		is_prefill = block_kv_cache is None

		b, t, d = x_in.shape

		q_latent = self.q_latent(x_in)
		query_latent = self.query(q_latent)
		query = rearrange(query_latent, 'b q (n d) -> b n q d', n=self.num_heads)

		kv_latent = self.kv_latent(x_in)

		if not is_prefill:
			kv_latent = torch.cat((block_kv_cache['kv_latent'], kv_latent), dim=1) # (B, T, low_rank) -> concat along the seq_len dimension
			# then (B, T, low_rank) * (low_rank, d_embed) -> (B, T, d_embed) which reassembles the full key, value matrices
		
		# reassemble full kv matrices (costs O(2*latent_dim*embed_dim))
		key_latent = self.key(kv_latent)
		value_latent = self.value(kv_latent)
		
		# reshape into heads
		key = rearrange(key_latent, 'b k (n d) -> b n k d', n=self.num_heads)
		value = rearrange(value_latent, 'b v (n d) -> b n v d', n=self.num_heads)

		# assemble causal self-attention matrix
		logits = torch.einsum('b n q d, b n k d -> b n q k', query, key) / ((d//self.num_heads)**0.5) # divide by the head_dim of the

		if is_prefill and attention_mask:
			logits = logits.masked_fill(~self.causal_mask[:t, :t], float('-inf')) # where the masked_fill isn't true, replace it with -inf

		scores = torch.softmax(logits, dim=-1)
		attention = torch.einsum('b n q k, b n k d -> b n q d', scores, value)
		attention = rearrange(attention, 'b n q d -> b q (n d)')
		
		attention_output = self.w_out(attention)

		block_kv_cache = {
			"kv_latent": kv_latent,
		}
		
		return attention_output, block_kv_cache

mla = MLA()
x_in = torch.rand(32,1,256)
x_out, kv_cache = mla(x_in)
print(x_out.shape)
print(kv_cache['kv_latent'].shape) # saved as (B, T, low_rank)

torch.Size([32, 1, 256])
torch.Size([32, 1, 4])


## MLA with RoPE embeddings (and better KV cache implementation)

- first find how many full rotations we do in max_seq_len (freq_cis), see: pos_encodings.ipynb for more

$$ i = dim \times log( \text{max seq len} / (\text{num rotations} \times 2\pi) ) / ( 2 \times log(base) ) $$

**Attention sinks**

- due to the softmax operation, sometimes models cannot evenly distribute the full probability (that must sum to 1) over all tokens, so it 'dumps' probability in the first 1-4 tokens
- while the first 1-4 tokens do not actually provide any semantic value, they basically act as a 'sink' for probability

- removing the attention sink is highly problematic and leads to a spike in perplexity (the model begins to assign uniform probability to everything), since the model has learned during training to use the sink to basically 'store excess probability mass'

This is a key consideration in a few attention mechanisms
- sliding window attention (we always keep the first 1-4 tokens in context) - see: [StreamingLLM](https://arxiv.org/pdf/2309.17453)
- MLA + RoPE - the first token is even more important as it acts as an un-rotated geometric anchor for the model to interpret positional encodings

In [15]:
def apply_rotary_emb(x_in: torch.Tensor, start_pos:int, d_embed:int, base:int=1e4): 

	shape = x_in.shape

	pair_index = torch.arange(0, d_embed, 2) # (d_embed//2,)

	positions = torch.arange(start_pos, start_pos+shape[-2]).unsqueeze(-1).float() # (max_seq_len,)
	
	inv_freq = torch.exp(-math.log(base) * (pair_index / d_embed)) # 1 / (10000^ (2i/d))

	freqs = positions * inv_freq

	pos_embeddings = torch.polar(torch.ones_like(freqs), freqs) # magnitude of the embeddings, (T, d_embed//2)

	### apply positional embeddings to a vector 
	x = torch.view_as_complex(x_in.float().contiguous().view(*shape[:-1], -1, 2)) # first split last dimension into 2, then merge using view_as_complex (means that you get cos(dim0)i + sin(dim1)j)
	y = torch.view_as_real(x * pos_embeddings).flatten(-2, -1) # then merge them back after polar multiplication

	return y.type_as(x_in)

In [20]:
from dataclasses import dataclass
import torch
import torch.nn as nn
from einops import rearrange
import math
import torch.nn.functional as F

@dataclass
class ModelArgs:

	qk_nope_head_dim: int = 128
	qk_rope_head_dim: int = 64
	v_head_dim: int=64
	q_lora_rank:int=96
	kv_lora_rank:int=64
	num_heads:int=8

	d_embed:int=256
	max_seq_len:int=1024
	max_batch_size:int=64

class RMSNorm(nn.Module):

	def __init__(self, d_embed: int, eps: float=1e-6):
		super().__init__()

		self.eps = eps
		self.weight = nn.Parameter(torch.ones(d_embed))

	def forward(self, x_in): 

		variance = x_in.pow(2).mean(-1, keepdim=True) # sum of the squares of feature dimension numbers

		x_norm = torch.rsqrt(variance + self.eps) # root square root of the variance to get standard deviation

		return x_in * x_norm * self.weight

class MLA(nn.Module): 

	def __init__(self, args: ModelArgs): 
		super().__init__()

		self.qk_head_dim = args.qk_nope_head_dim + args.qk_rope_head_dim # bigger up proj
		self.qk_nope_head_dim = args.qk_nope_head_dim
		self.qk_rope_head_dim = args.qk_rope_head_dim
		self.v_head_dim = args.v_head_dim
		self.d_embed = args.d_embed
		self.max_seq_len = args.max_seq_len
		self.max_batch_size = args.max_batch_size
		self.num_heads = args.num_heads
		self.kv_lora_rank = args.kv_lora_rank

		self.wq_a = nn.Linear(self.d_embed, args.q_lora_rank) # down projection into q_lora
		self.q_norm = RMSNorm(args.q_lora_rank)
		self.wq_b = nn.Linear(args.q_lora_rank, self.num_heads * (self.qk_nope_head_dim+self.qk_rope_head_dim)) # each head will have part of its dimensions as a PE and NOPE embedding 

		self.wkv_a = nn.Linear(self.d_embed, (args.kv_lora_rank + self.qk_rope_head_dim)) # down projection to kv latent - we cache this after splitting off the qk_rope_head_dim
		self.kv_norm = RMSNorm(args.kv_lora_rank)
		self.wkv_b = nn.Linear(args.kv_lora_rank, self.num_heads * (self.qk_nope_head_dim + self.v_head_dim), bias=False) # up projection gets both the k_NOPE and value matrices

		self.w_out = nn.Linear(self.num_heads * args.v_head_dim, self.d_embed) # project back to d_embed
		
		self.register_buffer('causal_mask', torch.tril(torch.ones(self.max_seq_len, self.max_seq_len)).bool(), persistent=False) # this is a nn.Module method that registers a self.causal_mask but on CUDA

		self.register_buffer('kv_cache', torch.zeros(self.max_batch_size, self.max_seq_len, args.kv_lora_rank)) # cache the kv of the down proj kv matrix
		self.register_buffer('pe_cache', torch.zeros(self.max_batch_size, self.max_seq_len, self.qk_rope_head_dim)) # cache the positional embeddings on the up proj of the key matrix

	def forward(self, x_in:int, is_prefill:bool=False, start_pos:int=0): 

		b, t, d = x_in.shape

		end_pos = start_pos + t 

		q_latent = self.wq_a(x_in)
		q_latent = self.q_norm(q_latent)
		q_proj = self.wq_b(q_latent)

		q_proj = rearrange(q_proj, 'b t (n d) -> b n t d', n=self.num_heads)

		# the up-projection of q will split into rope and nope head dims 
		q_nope, q_pe = torch.split(q_proj, [self.qk_nope_head_dim, self.qk_rope_head_dim], dim=-1) # q_nope is (b, n, t, qk_nope_head_dim)
		q_pe = apply_rotary_emb(q_pe, start_pos=start_pos, d_embed=self.qk_rope_head_dim) # q_pe is (b, n, t, qk_rope_head_dim)

		kv_latent = self.wkv_a(x_in)
		
		# notice k_pe has only one head per token instead of many as our q_pe does! 
		# basically we are doing MQA - where k_pe is our shared key head for the positional embeddings
		kv_nope, k_pe = torch.split(kv_latent, [self.kv_lora_rank, self.qk_rope_head_dim], dim=-1) # k_pe is (B, T, qk_rope_head_dim)
		kv_nope = self.kv_norm(kv_nope)
		k_pe = apply_rotary_emb(k_pe, start_pos=start_pos, d_embed=self.qk_rope_head_dim) #  k_pe is (b, t, qk_rope_head_dim)

		self.kv_cache[:b, start_pos:end_pos] = kv_nope # (B, T, kv_lora_rank), caching the normalized latent
		self.pe_cache[:b, start_pos:end_pos] = k_pe # (B, T, qk_rope_head_dim)

		if is_prefill:
			
			kv_proj = self.wkv_b(kv_nope)
			kv_proj = rearrange(kv_proj, 'b t (n d) -> b n t d', n=self.num_heads) # up projection to multi-head value and NOPE key matrices
			# compute QK
			q = torch.cat([q_nope, q_pe], dim=-1)
			k_nope, v = torch.split(kv_proj, [self.qk_nope_head_dim, self.v_head_dim], dim=-1) # 2x (B, num_heads, T, head_dim)

			# unsqueeze the positional embeddings along dim=1 so they can broadcast against the query heads later, also to match qk_nope's shape
			k = torch.cat([k_nope, k_pe.unsqueeze(1).expand(-1, self.num_heads, -1, -1)], dim=-1) # concatenate along embedding dim, so final is (B, T, num_heads, qk_nope + qk_rope)

			qk = torch.einsum('b n q d, b n k d -> b n q k', q, k) * (self.qk_head_dim**-0.5)
			qk = qk.masked_fill_(~self.causal_mask[start_pos:end_pos, :end_pos], float('-inf'))

			scores = F.softmax(qk, dim=-1)

			x = torch.einsum('b n q k, b n k d -> b n q d', scores, v) 
			x = rearrange(x, 'b n q d -> b q (n d)', n=self.num_heads) 

		else:
			wkv_b = self.wkv_b.weight
			wkv_b = wkv_b.view(self.num_heads, -1, self.kv_lora_rank) # this is a splitting from (num_heads * head_dim, kv_lora_rank) => (num_heads, per_head_dim, kv_lora_rank)
			
			# we do the latent trick - multiply q_nope (B, num_heads, T, qk_nope_head_dim) by down projection matrix (num_heads, qk_nope_head_dim, kv_lora_rank)
			qk_latent = torch.einsum('b n q d, n d k -> b n q k', q_nope, wkv_b[:, :self.qk_nope_head_dim]) # (B, num_heads, T, kv_lora_rank)

			### NOTE: this is a bit of a confusing operation above, here's how you'd do it normally
			# wkv_b = wkv_b[:, :self.qk_nope_head_dim]
			# q_nope @ wkv_b.unsqueeze(1) # this broadcasts along the sequence dimension, applying the latent compression to the q_nope embeddings

			qk = torch.einsum('b n q c, b t c -> b n q t', qk_latent, self.kv_cache[:b, :end_pos]) # (B, num_heads, T, kv_lora_rank) @ (B, T, kv_lora_rank)
			
			scores = (qk + torch.einsum('b n q c, b t c -> b n q t', q_pe, self.pe_cache[:b, :end_pos])) * (self.qk_head_dim ** -0.5) # (B, num_heads, seq_len, v_head_dim)

			scores = scores.softmax(dim=-1)

			x = torch.einsum('b n q t, b t c -> b n q c', scores, self.kv_cache[:b, :end_pos]) # projects down to the kv_cache_dim
			x = torch.einsum('b n q c, n d c -> b n q d', x, wkv_b[:, -self.v_head_dim:]) # projects back up from kv_lora_rank to v_head_dim
			x = rearrange(x, 'b n q d -> b q (n d)', n=self.num_heads)

			### NOTE: notice that we always perform calculations in the latent dim, then project upwards to the dim we want. Both in attention and the value matrix - never computing the full matrix for V (B, T, v_head_dim), or the full K matrix (B, T, qk_nope_head_dim) - during DECODE

		attention_output = self.w_out(x)
		return attention_output

model_args = ModelArgs()
mla = MLA(model_args)
x_in = torch.rand(32,1024,256)
x_out = mla(x_in, is_prefill=True)
x_out.shape

torch.Size([32, 1024, 256])

## Arithmetic intensity of prefill

From: https://arxiv.org/pdf/1911.02150

Compare how many FLOPs are done, compared to memory access operations (which is the number of tensors involved in the operation) -> this is fundamentally what flash attention fixes

Arithmetic intensity = divide **FLOPs** by **memory access** (i.e. FLOPs per byte moved)

- _higher is better_ - you want many FLOPs per byte, so the GPU is computing instead of sitting idle waiting on HBM
- compare it against the hardware's own ops:byte ratio (~100:1 on a modern GPU). Below that line you are memory-bound

**Assumption throughout: $n > d$** (long context), so the $n^2$ attention terms dominate

**MHA prefill stage**

_comparing just the self-attention operation, not assembling of QKV matrices_

- FLOPs =

  $$\Theta(bn^2d)$$
  - the two $n \times n$ einsums: $QK^T$ and $\text{scores} \times V$
  - (if $n < d$ this would instead be $bnd^2$, dominated by the QKV projections)

- Memory access =
  $$O(bnd + bhn^2 + d^2)$$
  - $3 \times bnd$ - the output of the QKV matrices, the input X and the output Y
  - $bhn^2$ - the self-attention matrix (per head)
  - $d^2$ the final w_out matrix
- Arithmetic intensity
  $$\frac{\Theta(bn^2d)}{O(bnd + bhn^2 + d^2)} \rightarrow \Theta(k)$$
  - with $n > d$ the $bhn^2$ score matrix dominates the denominator, so the ratio collapses to $\frac{bn^2d}{bhn^2} = \frac{d}{h} = k$, the **head dim**
  - i.e. intensity is _capped by the head dimension_ no matter how long the sequence gets, because the $n^2$ score matrix gets written out to HBM and then read straight back in
  - flash attention fixes exactly this: it never materializes the $n^2$ matrix, killing the $bhn^2$ term so intensity is no longer pinned to $k$

  **NOTE:** Shazeer assumes $n \le d$ (in 2019, $d_{model}$ = 1024 with sequences of 256), so his FLOPs are $\Theta(bnd^2)$ - dominated by the QKV/output projections rather than by the attention itself. He also states the **reciprocal**, memory ÷ compute: $O(\frac{1}{k} + \frac{1}{bn})$ - written that way so _lower is better_ and "close to 1" (one byte moved per FLOP) is the memory-bound alarm bell. Same physics, flipped fraction.


## Arithmetic intensity of decode

**MHA decode stage**

_Complexity for generating n tokens using decode below, still assuming $n > d$_

- FLOPs = $$\Theta(bn^2d)$$
  - self-attention is cheap since its $Q$ is [1, d_embed], $K^T$ is [d_embed, seq_len]
  - we do this n times, thus $bnd \times n = bn^2d$

- Memory access = $$O(bn^2d + nd^2)$$
  - Query matrix = $b, 1, d$
  - Key and value matrix = $b, n, d$
  - Self-attention matrix = $b, 1, n$
  - W_out matrix = $d, d$
  - all of this multiplied by $n$, for generating $n$ tokens - the cost is dominated by the KV matrices

- Arithmetic intensity = $$\frac{\Theta(bn^2d)}{O(bn^2d + nd^2)} = \Theta(\frac{1}{\frac{d}{bn} + 1})$$
  - both numerator and denominator are dominated by the same $bn^2d$ term, so as $n$ grows the ratio **saturates at $\Theta(1)$** - it does not keep falling
  - but $\Theta(1)$ is a terrible place to sit: ~1 FLOP per byte moved, against hardware that wants ~100. Decode is memory-bandwidth-bound _no matter how long the context is_, and with $b = 1$ we never climb out
  - contrast with prefill, which reaches $\Theta(k)$ - decode is fundamentally worse because each step reads the whole KV cache to do a single token's worth of math
  - the greatest cost is loading the KV cache every time we decode a token $O(2bnd)$
  - hence the development of $MQA, GQA, MLA$ to compress the KV cache - they shrink the memory term, which is the only lever that moves this ratio
  - the other lever is $b$: batching more sequences amortizes the same KV read over more queries, which is why serving systems batch aggressively


Linear attention

some intuition on why linear attention > dot production attention in time complexity

<img src="https://www.changjiangcai.com/mystudynotes/docs/auto-encoding/images/78_annotated-diffusion/linear-attention.png" width="600">

What's happening

Full blog: https://haileyschoelkopf.github.io/blog/2024/linear-attn/

- instead of materializing the full $QK^T$ matrices, we instead:
  - get rid of softmax that binds Q and K together and use a kernel approximation instead of softmax, that mirrors the feature mapping (definitions below), but allows Q and K to be separable

$$\text{Softmax}\left(QK^T\right)V = \frac{ \text{exp}(QK^T) } {\sum_{i=1}^L \text{exp}(QK_i^T)} V$$

- specifically we use $\phi(x) = ELU(x) + 1$, where $ELU = e^x-1 $ for $x≤0$ and x for $x≥0$

- and we normalize ELU by $\phi(Q) \phi(K^T)$ to get us back to probs like softmax

$$ \text{Linear Attention}\left(QK^T\right)V = \frac{ \phi(Q)\phi(K)^T } {\sum\_{i=1}^L \phi(Q)\phi(K_i)^T} V $$

Using the associativity of matrix multiplication:
$$ \frac{ \phi(Q)\phi(K)^T } {\sum*{i=1}^L \phi(Q)\phi(K_i)^T} V = \frac{ \phi(Q)(\phi(K)^T V)} {\phi(Q)\sum*{i=1}^L \phi(K_i)^T} $$

- this allows us to do $ S = \phi(K^T) V^T$ which creates a state space matrix of size $d \times d$, lets call this KV
- we update the S like this: $S_{t+1} = S_t + K_{t+1}^T V_{t+1}$
- by recursion $S_t = \sum_{i=1}^{t}(\phi(K_i^T)V_i)$, where t is token number t in the sequence
- i.e. it is the sum of all K and V vectors from token 1 -> t
  - the query $Q_t$ attends to this $d \times d$ KV matrix, which contains all the state knowledge of all $t$ tokens (thus, state space matrix)

_Massive efficiencies in compute time_

- Computational complexity: $O(n)$, n = seq_len
- Memory complexity: $O(d^2)$, does not scale with seq_len
- Arithmetic intensity: $O(\dfrac{n}{d^2})$

- Flaws:
  - this is great for decode, every new query can attend to this $d \times d$ state space matrix
    - decode is $O(1)$, since its just $d^2$ which doesn't grow (query vector 1xd, attending to S that is dxd)
  - this is terrible for prefill, since every token needs to attend to the $d \times d$ only composed of the current + previous tokens' KV vectors
    - this is the problem of recurrence which does not mesh well with GPUs excellent parallel computational powers, which self-attention fully exploits

    - despite being $O(N)$, the sequential nature of materializing $S_n$ makes it hard to use

- naive linear attention, no sequence chunking and parallelism
- linear attention is O(N)

Definitions

- feature map = an element-wise function that takes an input and maps it to a new set of features, typically in higher dimensions, but in this case, non-negative space

- kernel approximation = distinct from kernel functions, its basically a function that takes otherwise non-linear computation, maps it to a linear space. In this case we are mimicking O(N^2) softmax attention to O(N) linear attention


In [148]:
class LinearAttention(nn.Module): 

	def __init__(self, d_embed:int=256):

		super().__init__()

		self.q = nn.Linear(d_embed, d_embed)
		self.k = nn.Linear(d_embed, d_embed)
		self.v = nn.Linear(d_embed, d_embed)

	def forward(self, x_in):
		
		b, t, d = x_in.shape
		
		tokens = torch.split(x_in, split_size_or_sections=1, dim=1)

		# recurrence
		kv = torch.zeros(b, d, d, device=x_in.device, dtype=x_in.dtype)
		z  = torch.zeros(b, d, device=x_in.device, dtype=x_in.dtype) # normalizer
		# we want to replicate softmax, which normalizes along dim=-1 of the qk matrices, ensuring magnitudes sum to 1 (right stochastic matrix)
		# so we need to divide the final output, by the magnitude of the qk matrix across rows
		
		token_activations = []
		for token in tokens:
			
			elu = torch.nn.ELU()
			
			q = elu(self.q(token))+1 # (b, 1, d)
			k = elu(self.k(token))+1 

			z += k.squeeze(1) # (b,d) -> that is how much we should normalize the columns of k by, recall there are d columns per sample 

			v = self.v(token)

			kv += torch.einsum('bnk, bnv -> bkv', k, v) # add to the previous S_t to get S_{t+1}
			attention_out = torch.einsum('bqk, bkv -> bqv', q, kv)
			denom = torch.einsum('bnk, bk -> bn', q, z) # (b, t, d) @ (b, d) -> (b, 1), summing the magnitude of the row of q with the magnitude of the column in k, giving the magnitude of the token

			token_activations.append(attention_out / (denom.unsqueeze(-1) + 1e-6)) # normalize with laplace smoothing

		return torch.cat(token_activations, dim=1) # along the sequence length dimension, concatenate

x_in = torch.rand(32, 1024, 256)
linear = LinearAttention(d_embed=256)
out = linear(x_in)
out.shape

torch.Size([32, 1024, 256])

Linear attention with efficient chunking

_much faster prefill_

- with chunking, here we chunk up the tokens equally

- then we compute the KV matrix (i.e. d x d state space matrix) for each chunk, adding on the values from the previous chunk
  - i.e. $S_{t+1} = S_t + K_{t+1}^T V_{t+1}$

- within the current chunk we use causal self-attention (but keeping ELU as our kernel approximation for the feature map performed by softmax attention)

- $o_{c+1} = (Q_{c+1} KV_c + Q_{c+1} K_{c+1}^T \odot Mask ) V_{c+1}$
  - the new query chunk attends to the previous KV chunks, where KV*t = sum of all KV*{1->t}
  - then we also want this query to attend to the other keys and values in this chunk, so we use causal self-attention


Deepseek sparse attention

- used by DS-v3

<img src="https://sebastianraschka.com/llm-architecture-gallery/images/concepts/deepseek-sparse-attention-comparison.webp" width="600">

<img src="https://sebastianraschka.com/llm-architecture-gallery/images/concepts/deepseek-sparse-attention-flow.webp" width="600">
